In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import conntility
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import bosperrus

sys.path.insert(0, str((Path("../..") / "src" / "physically_realized_graphs").resolve()))
import micron_comparison

plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
BSPBLUE = bosperrus.PiecewiseLinearFit.color

MICRONS_FN_MAT = "/home/woody/iwbn/iwbn007h/bosperrus/truncated_graphs/data/reimann_data/microns_mm3_connectome.h5"
NBINS = 51             # x-z grid resolution for the Reimann outer-synapse-fraction heuristic
ALPHA_SHAPE = 0.0001    # concaveness of the alpha shape fit to the neuron x-z footprint
REIMANN_THRESH = 0.05  # outer-synapse-fraction cutoff for Reimann's "affected" classification

SAVE_PDF = "../../result_plots/physically_realized_graphs/microns.pdf"
SAVE_PNG = "../../../../bosperrus/rendered_figures/microns.png"

In [ ]:
microns_data = micron_comparison.prepare_microns_data(
    MICRONS_FN_MAT, nbins=NBINS, alpha=ALPHA_SHAPE, reimann_thresh=REIMANN_THRESH,
)
micron_comparison.print_comparison_summary(microns_data["comparison"])

In [ ]:
# ── data preparation ──────────────────────────────────────────────────────────
def load_connectome(fn_mat, dataset="full"):
    """Load the MICrONS connectome from an h5 file via conntility."""
    return conntility.ConnectivityMatrix.from_h5(fn_mat, dataset)

In [ ]:
microns_original = load_connectome(MICRONS_FN_MAT, dataset="full")

In [ ]:
microns_original.vertex_properties

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3), width_ratios=[2, 3])

micron_comparison.plot_fits_panel(
    axs[0],
    microns_data["flow"],
    microns_data["alpha_distances"],
    microns_data["scores"],
    measure="degree",
    fit_color=BSPBLUE,
    legend_label_fn=lambda measure, b, m, c: f"piecewise-linear (elbow={b/1e3:.2f})",
    ylabel="Degree",
    title="MICrONS mm³ connectome",
)

micron_comparison.plot_boundary_heatmap_panel(
    axs[1], microns_data,
    reimann_thresh=REIMANN_THRESH,
    cmap="gray",
    boundary_label="Alpha shape", boundary_color="red",
    reimann_neuron_label="Manual 5% (per neuron)", reimann_neuron_color="#7FBF7B",
    reimann_bin_label="Manual 5% (per bin max)", reimann_bin_color="#1B7837",
    elbow_color=BSPBLUE,
    legend_bbox_to_anchor=(0.5, -0.65),
)

os.makedirs(os.path.dirname(SAVE_PDF), exist_ok=True)
fig.savefig(SAVE_PDF, bbox_inches="tight")
fig.savefig(SAVE_PNG, bbox_inches="tight")


plt.show()